# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets by @id
record_set_ids = [rs['@id'] for rs in metadata.to_json().get('recordSet', [])]

print('Available Record Sets:')
for rs in metadata.to_json().get('recordSet', []):
    print(f"- @id: {rs['@id']}, name: {rs.get('name', 'N/A')}, description: {rs.get('description', 'N/A')}")

# For demonstration: Show details of the first record set
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    # Show fields of the first record set
    example_record_set = [x for x in metadata.to_json().get('recordSet', []) if x['@id'] == example_record_set_id][0]
    print(f"\nFields in Record Set @{example_record_set_id}:")
    for f in example_record_set.get('field', []):
        field_name = f.get('name', 'N/A') if isinstance(f, dict) else str(f)
        field_id = f['@id'] if isinstance(f, dict) and '@id' in f else (f if isinstance(f, str) else 'N/A')
        print(f"- @id: {field_id}, name: {field_name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# If there are no record sets, halt here gracefully
dataframes = {}
if not record_set_ids:
    print('No record sets defined in the schema. Please check the Croissant schema definitions for available data.')
else:
    # Extract data from all available record sets
    for record_set_id in record_set_ids:
        try:
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for record set @id: {record_set_id}")
        except Exception as e:
            print(f"Could not load records for {record_set_id}: {e}")

    # Display columns of the first record set if loaded
    if record_set_ids[0] in dataframes:
        print(f"\nColumns in DataFrame for @{record_set_ids[0]}:")
        print(dataframes[record_set_ids[0]].columns.tolist())
        display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA: Filtering, normalizing, grouping for a numeric field in the first available record set.
import numpy as np
if dataframes:
    main_record_set_id = record_set_ids[0]
    df = dataframes[main_record_set_id].copy()

    # Try to find a numeric field automatically
    # We look for common numeric field name hints used in proteomics datasets
    possible_numeric_fields = [col for col in df.columns if any(substr in col.lower() for substr in ['abundance', 'mw', 'weight', 'mean', 'count', 'coverage'])]
    if possible_numeric_fields:
        numeric_field = possible_numeric_fields[0]
        print(f"Using numeric field: '{numeric_field}' for EDA.")

        try:
            # Cast to float for safety
            df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
            # Drop NaN for EDA
            df = df.dropna(subset=[numeric_field])
            threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 10
            filtered_df = df[df[numeric_field] > threshold]
            print(f"Filtered records with {numeric_field} > {threshold}\n")
            display(filtered_df.head())

            filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            print(f"\nNormalized '{numeric_field}' for filtered records:")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            # Attempt to group by 'description' or first string column
            string_cols = df.select_dtypes(include='object').columns.tolist()
            group_field = 'description' if 'description' in string_cols else (string_cols[0] if string_cols else None)
            if group_field:
                print(f"\nGrouping by '{group_field}': mean of '{numeric_field}'")
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
                display(grouped_df.head())
        except Exception as e:
            print(f"Error during EDA: {e}")
    else:
        print("No obvious numeric field found for EDA. Please inspect the DataFrame columns:")
        print(df.columns.tolist())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and possible_numeric_fields:
    num_field = possible_numeric_fields[0]
    df_viz = dataframes[main_record_set_id].copy()
    df_viz[num_field] = pd.to_numeric(df_viz[num_field], errors='coerce')
    df_viz = df_viz.dropna(subset=[num_field])

    plt.figure(figsize=(8, 4))
    sns.histplot(df_viz[num_field], kde=True)
    plt.title(f"Distribution of '{num_field}'")
    plt.xlabel(num_field)
    plt.ylabel('Frequency')
    plt.show()

    # If there's a second numeric field, scatter plot
    if len(possible_numeric_fields) > 1:
        y_field = possible_numeric_fields[1]
        df_viz[y_field] = pd.to_numeric(df_viz[y_field], errors='coerce')
        df_viz = df_viz.dropna(subset=[num_field, y_field])
        plt.figure(figsize=(6,6))
        sns.scatterplot(x=df_viz[num_field], y=df_viz[y_field])
        plt.title(f"{num_field} vs. {y_field}")
        plt.xlabel(num_field)
        plt.ylabel(y_field)
        plt.show()


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated how to use `mlcroissant` to load and explore a dataset following the Croissant schema.

- We loaded metadata and investigated available record sets and their fields using their `@id`s.
- We performed example extraction into pandas DataFrames.
- Exploratory data analysis (EDA) was conducted: filtering, normalization, and grouping on numeric fields.
- Visualizations provided insight into data distributions.

**Tip:** Always consult the Croissant schema documentation or dataset readme for authoritative field descriptions and intended usage.